In [65]:
import numpy as np
import pandas as pd
from cohort_analyzer import CohortAnalyzer
from scipy.stats import weibull_min

In [68]:
class ScenarioSimulator:

    def __init__(self, original_df, discount_rate = 0.10):
        self.original_df = original_df.copy()
        self.discount_rate = discount_rate
        self.scenarios= {}

        
    def create_scenario(self, name, arpu_multiplier= 1.0, retention_improvement = 0.0, 
                        cac_reduction= 0.0, channel_specific = None):
        
        modified_df = self.original_df.copy()
        modified_df = modified_df.astype({'arpu' : 'float64' , 'cac' : 'float64' , 'monthly_revenue' : 'float64'})

        if arpu_multiplier != 1.0:
            modified_df['arpu'] = modified_df['arpu'].astype(float) * arpu_multiplier
            modified_df['monthly_revenue'] = modified_df['monthly_revenue'] * arpu_multiplier

        if cac_reduction != 0.0:
            modified_df['cac'] = modified_df['cac'] * (1 - cac_reduction)

        if retention_improvement != 0.0:
            modified_df['weibull_scale']= modified_df['weibull_scale'] * ( 1+ retention_improvement)

        if retention_improvement != 0.0:
            for idx, row in modified_df.iterrows():
                shape = row['weibull_shape']
                scale= row['weibull_scale']
                month = row['months_since_signup']

                new_retention = np.exp(-(month/scale) ** shape)
                modified_df.at[idx, 'retention_rate'] = new_retention
                modified_df.at[idx, 'active_users'] = int(row['cohort_size'] * new_retention)
                modified_df.at[idx, 'monthly_revenue'] = row['arpu'] * modified_df.at[idx, 'active_users']



        if channel_specific:
            for channel, params in channel_specific.items():
                mask = modified_df['channel'] == channel

                if 'arpu_multiplier' in params:
                    mult = params['arpu_multiplier']
                    modified_df.loc[mask, 'arpu'] = modified_df.loc[mask, 'arpu'] * mult
                    modified_df.loc[mask, 'monthly_revenue'] = modified_df.loc[mask, 'active_users'] * modified_df.loc[mask, 'arpu']

                if 'cac_reduction' in params:
                    reduction = params['cac_reduction']
                    modified_df.loc[mask, 'cac'] = modified_df.loc[mask, 'cac'] * (1 - reduction)

                if 'retention_improvement' in params:
                    improvement = params['retention_improvement']
                    modified_df.loc[mask, 'weibull_shape'] = (
                        modified_df.loc[mask, 'weibull_shape'] + improvement
                    )

                    for idx in modified_df[mask].index:
                        row = modified_df.loc[idx]
                        shape = row['weibull_shape']
                        scale = row['weibull_scale']
                        month = row['months_since_signup']

                        new_retention = np.exp(-(month/ scale) ** shape)
                        modified_df.at[idx, 'retention_rate'] = new_retention
                        modified_df.at[idx, 'active_users'] =  int(row['cohort_size'] * new_retention)
                        modified_df.at[idx, 'monthly_revenue'] = row['arpu'] * modified_df.at[idx, 'active_users']
                        
        self.scenarios[name] = {
            'df' : modified_df,
            'params' : {
                'arpu_multiplier': arpu_multiplier,
                'retention_improvement': retention_improvement,
                'cac_reduction' : cac_reduction,
                'channel_specific' : channel_specific
            }
        }

        return self

    def get_scenario_ltv(self, scenario_name):
        if scenario_name not in self.scenarios:
            raise ValueError(f"Scenario '{scenario_name}' not found")

        df = self.scenarios[scenario_name]['df']
        analyzer = CohortAnalyzer(df, discount_rate = self.discount_rate)
        return analyzer.calculate_ltv_by_cohort_channel()

    def compare_scenarios(self, scenarios = None):
        if scenarios is None:
            scenarios = list(self.scenarios.keys())

        original_analyzer = CohortAnalyzer(self.original_df, discount_rate = self.discount_rate)
        original_ltv = original_analyzer.calculate_ltv_by_cohort_channel()

        comparison = []

        
        for _, row in original_ltv.groupby('channel').agg({
            'cac': 'first',
            'ltv' : 'mean',
            'ltv_ratio' : 'mean',
            'payback_month' : 'mean'

        }).reset_index().iterrows():
            comparison.append({
                'scenario' : 'BASELINE (Original)',
                'channel' : row['channel'],
                'avg_cac' : row['cac'],
                'avg_ltv' : round(row['ltv'], 2),
                'avg_ltv_ratio' : round(row['ltv_ratio'], 2),
                'avg_payback_months' : round(row['payback_month'], 1)
            })

        for scenario_name in scenarios:
            if scenario_name not in self.scenarios:
                print(f"Warning: Scenario '{scenario_name}' not found, skipping")
                continue

            ltv_df = self.get_scenario_ltv(scenario_name)
            params = self.scenarios[scenario_name]['params']

            for _, row in ltv_df.groupby('channel').agg({
                'cac' : 'first',
                'ltv' : 'mean',
                'ltv_ratio' : 'mean',
                'payback_month' : 'mean'
            }).reset_index().iterrows():
                comparison.append({
                    'scenario' : scenario_name,
                    'channel' : row['channel'],
                    'avg_cac' : row['cac'],
                    'avg_ltv' : round(row['ltv'], 2),
                    'avg_ltv_ratio' : round(row ['ltv_ratio'], 2),
                    'avg_payback_months' : round(row['payback_month'], 1)

                })

        comparison_df = pd.DataFrame(comparison)
        return comparison_df

    def impact_analysis(self, scenario_name):
        original_analyzer = CohortAnalyzer(self.original_df, discount_rate= self.discount_rate)
        original_ltv = original_analyzer.calculate_ltv_by_cohort_channel()

        scenario_ltv = self.get_scenario_ltv(scenario_name)

        impact = []

        for channel in original_ltv['channel'].unique():
            orig = original_ltv[original_ltv['channel'] == channel]
            scen = scenario_ltv[scenario_ltv['channel'] == channel]

            if len(orig) == 0 or len(scen) == 0:
                continue

            orig_avg_ltv = orig['ltv'].mean()
            scen_avg_ltv = scen['ltv'].mean()
            orig_avg_ratio = orig['ltv_ratio'].mean()
            scen_avg_ratio = scen['ltv_ratio'].mean()
            orig_payback = orig['payback_month'].mean()
            scen_payback = scen['payback_month'].mean()
            
            ltv_change_pct = ((scen_avg_ltv - orig_avg_ltv) / orig_avg_ltv * 100) if orig_avg_ltv > 0 else 0
            ratio_change_pct = ((scen_avg_ratio - orig_avg_ratio) / orig_avg_ratio * 100) if orig_avg_ratio > 0 else 0   
            payback_change_months = (scen_payback - orig_payback) if not np.isnan(orig_payback) else 0


            
            impact.append({
                'channel' : channel,
                'baseline_ltv' : round(orig_avg_ltv, 2),
                'scenario_ltv' : round(scen_avg_ltv, 2),
                'ltv_change_%' :round(ltv_change_pct, 1),
                'baseline_ltv_ratio' : round(orig_avg_ratio, 2),
                'scenario_ltv_ratio' : round(scen_avg_ratio, 2),
                'ratio_change_%' : round(ratio_change_pct, 1),
                'baseline_payback_months': round(orig_payback, 1),
                'scenario_payback_months' : round(scen_payback, 1),
                'payback_change_months' : round(payback_change_months, 1)

            })

        return pd.DataFrame(impact)

    def print_scenario_comparisons(self, scenarios= None):
        comparison = self.compare_scenarios(scenarios)

        print("\n" + "-"*100)
        print("SCENARIO COMPARISON - CHANNEL PROFOTABILITY")
        print("\n" + "-"*100)


        for scenario in comparison['scenario'].unique():
            scenario_data = comparison[comparison['scenario'] == scenario]
            print(f"\n{scenario.upper()}")
            print("-" * 100)
            print(scenario_data.to_string(index = False))
                  
    def print_impact_analysis(self, scenario_name):
                
        impact = self.impact_analysis(scenario_name)

        print("\n" + "-" * 120)
        print(f"IMPACT ANALYSIS: {scenario_name.upper()}")
        print("-" * 120)
        print(impact.to_string(index = False))
        print("-" * 120 + "\n")
       

In [69]:
if __name__ == "__main__":
    df = pd.read_csv('synthetic_cohorts.csv')

    sim = ScenarioSimulator(df, discount_rate = 0.10)

    sim.create_scenario("Raise ARPU 20 %", arpu_multiplier = 1.2)

    sim.create_scenario("Improve Retention 10%", retention_improvement = 0.1)

    sim.create_scenario("Reduce CAC 15% + Raise ARPU 10%", arpu_multiplier = 1.1, cac_reduction = 0.15)

    sim.create_scenario("Double down on Organic (ARPU + 30%, CAC -20%)", channel_specific = {'organic' : {'arpu_multiplier' : 1.3, 'cac_reduction' : 0.2}})

    sim.print_scenario_comparisons()

    for scenario in list(sim.scenarios.keys())[:2]:
        sim.print_impact_analysis(scenario)   


----------------------------------------------------------------------------------------------------
SCENARIO COMPARISON - CHANNEL PROFOTABILITY

----------------------------------------------------------------------------------------------------

BASELINE (ORIGINAL)
----------------------------------------------------------------------------------------------------
           scenario     channel  avg_cac   avg_ltv  avg_ltv_ratio  avg_payback_months
BASELINE (Original)      direct     35.0  80869.72        2310.56                 0.0
BASELINE (Original)     organic     20.0 124976.45        6248.82                 0.0
BASELINE (Original) paid_search     45.0  79394.37        1764.32                 0.0
BASELINE (Original) paid_social     70.0  33020.29         471.72                 0.0
BASELINE (Original)    referral     25.0  56531.43        2261.26                 0.0

RAISE ARPU 20 %
-------------------------------------------------------------------------------------------------

In [64]:
import inspect
print(inspect.signature(ScenarioSimulator.create_scenario))

(self, name, arpu_multiplier=1.0, retention_improvement=0.0, cac_reduction=0.0, channel_specific=None)
